In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, MapType

warehouse_path = "/tmp/ehdip_warehouse"
spark = SparkSession.builder \
    .appName("Day03_Batch_CDC") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", warehouse_path) \
    .getOrCreate()

spark.sql("CREATE NAMESPACE IF NOT EXISTS local.ehdip_bronze")
spark.sql("""
CREATE TABLE IF NOT EXISTS local.ehdip_bronze.raw_payloads (
    payload_id STRING,
    source_system_id STRING,
    raw_payload_json STRING,
    ingestion_timestamp TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
""")

# Simulate X12 Batch ingestion
x12_data = [("ISA*00*          *00*          *ZZ*1234567890     *ZZ*9876543210     *230510*1034*^*00501*000000001*0*T*:~",)]
x12_df = spark.createDataFrame(x12_data, ["value"])
x12_df.createOrReplaceTempView("x12_view")
spark.sql("""
INSERT INTO local.ehdip_bronze.raw_payloads
SELECT uuid() as payload_id, 'x12-batch' as source_system_id, value as raw_payload_json, current_timestamp() as ingestion_timestamp
FROM x12_view
""")

# Simulate Debezium CDC ingestion
cdc_data = [("c", None, '{"id": 1, "status": "active"}', {"db": "claims"})]
cdc_schema = StructType([
    StructField("op", StringType(), True),
    StructField("before", StringType(), True),
    StructField("after", StringType(), True),
    StructField("source", MapType(StringType(), StringType()), True)
])
cdc_df = spark.createDataFrame(cdc_data, schema=cdc_schema)
cdc_df.createOrReplaceTempView("cdc_view")
spark.sql("""
INSERT INTO local.ehdip_bronze.raw_payloads
SELECT uuid() as payload_id, 'debezium-cdc' as source_system_id, after as raw_payload_json, current_timestamp() as ingestion_timestamp
FROM cdc_view WHERE op IN ('c', 'u', 'd')
""")

spark.sql("SELECT source_system_id, raw_payload_json FROM local.ehdip_bronze.raw_payloads ORDER BY ingestion_timestamp DESC LIMIT 5").show(truncate=False)
